# Serving Hardening

## Notebook Contract
- **Objective:** demonstrate the v0.5 deployment-hardening additions to the FastAPI serving layer: liveness/readiness probes, request logging, model warm-up, an optional Prometheus metrics endpoint, and a `docker-compose` recipe.
- **Inputs:** an in-process `TestClient` driving `create_app` with an injected fake predictor (so the notebook stays offline; no model download).
- **Outputs:** captured structured request logs, a sample `/metrics` payload (when `prometheus-client` is installed), and a captured `/health/ready` failure mode written under `reports/sample_run/serving/`.
- **Expected runtime:** under 30 seconds on CPU.
- **Scope boundary:** all serving code lives in `src/hf_finetuning_lab/serving/`; this notebook orchestrates the demo via `TestClient`.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import json
import logging
import os
import platform
import random
import sys
from datetime import UTC, datetime
from io import StringIO
from pathlib import Path
from typing import Any

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from fastapi.testclient import TestClient

from hf_finetuning_lab.serving.api import create_app
from hf_finetuning_lab.serving.logging import REQUEST_LOGGER_NAME

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")

Python: 3.12.11
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-05-19T16:22:15+00:00


## 2) Parameters

In [2]:
SERVING_REPORTS = ROOT / 'reports' / 'sample_run' / 'serving'
SERVING_REPORTS.mkdir(parents=True, exist_ok=True)
LOG_DUMP_PATH = SERVING_REPORTS / 'request_log.jsonl'
METRICS_DUMP_PATH = SERVING_REPORTS / 'metrics_sample.txt'
READINESS_FAIL_PATH = SERVING_REPORTS / 'readiness_failure.json'

MODEL_DIR = ROOT / 'artifacts' / 'models' / 'support-triage'
MODEL_VERSION = '0.5.0-demo'
print(f'Reports dir: {SERVING_REPORTS}')
print(f'Model dir (advertised): {MODEL_DIR}')

Reports dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\serving
Model dir (advertised): C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\artifacts\models\support-triage


## 3) Inject a Fake Predictor

`create_app` accepts a `predictor_factory` so we can drive the API without loading a real transformer. The fake records every call (warm-up plus user requests) so we can verify the lifecycle.

In [3]:
class FakePredictor:
    def __init__(self) -> None:
        self.calls: list[list[str]] = []

    def predict(self, texts: list[str]) -> list[dict[str, Any]]:
        self.calls.append(list(texts))
        return [{'label': 'billing', 'score': 0.42} for _ in texts]


predictor = FakePredictor()
app = create_app(
    model_dir=MODEL_DIR,
    predictor_factory=lambda _path: predictor,
    warm_up_texts=('warm up the model', 'second warm-up call'),
    model_version=MODEL_VERSION,
)
print('App routes:')
for route in sorted(app.routes, key=lambda r: getattr(r, 'path', '')):
    methods = ','.join(sorted(getattr(route, 'methods', set()) or set()))
    print(f"  {getattr(route, 'path', '')}  [{methods}]")

App routes:
  /docs  [GET,HEAD]
  /docs/oauth2-redirect  [GET,HEAD]
  /health  [GET]
  /health/live  [GET]
  /health/ready  [GET]
  /openapi.json  [GET,HEAD]
  /predict  [POST]
  /redoc  [GET,HEAD]


## 4) Capture Structured Request Logs

Attach an in-memory log handler to the dedicated request logger so we can read every log line emitted while we drive the endpoints.

In [4]:
log_stream = StringIO()
log_handler = logging.StreamHandler(log_stream)
log_handler.setFormatter(logging.Formatter('%(message)s'))
request_logger = logging.getLogger(REQUEST_LOGGER_NAME)
request_logger.addHandler(log_handler)
request_logger.setLevel(logging.INFO)
print(f'Listening on logger: {REQUEST_LOGGER_NAME}')

Listening on logger: hf_finetuning_lab.serving.requests


## 5) Exercise the Endpoints

`TestClient` enters the app lifespan on `with client:`, so the warm-up call runs the first time we enter the block. After exit the predictor state is reset.

In [5]:
client = TestClient(app)
responses: list[dict[str, Any]] = []
with client:
    for method, path, json_body in (
        ('GET', '/health', None),
        ('GET', '/health/live', None),
        ('GET', '/health/ready', None),
        ('POST', '/predict', {'texts': ['I cannot log in', 'My invoice is wrong']}),
        ('POST', '/predict', {'texts': []}),
    ):
        if method == 'GET':
            response = client.get(path)
        else:
            response = client.post(path, json=json_body)
        try:
            body = response.json()
        except ValueError:
            body = response.text
        responses.append(
            {
                'method': method,
                'path': path,
                'status': response.status_code,
                'body': body,
            }
        )
request_logger.removeHandler(log_handler)
pd.DataFrame(responses)

,method,path,status,body
0,GET,/health,200,"{'status': 'ok', 'model_dir': 'C:\Users\diogo\..."
1,GET,/health/live,200,"{'status': 'ok', 'model_dir': 'C:\Users\diogo\..."
2,GET,/health/ready,200,"{'status': 'ready', 'model_dir': 'C:\Users\dio..."
3,POST,/predict,200,"{'predictions': [{'label': 'billing', 'score':..."
4,POST,/predict,422,"{'detail': [{'type': 'too_short', 'loc': ['bod..."


## 6) Warm-Up Evidence

The predictor records every call. The first two entries are the warm-up texts; user requests appear afterwards. This is the contract that keeps the first user request off the cold path.

In [6]:
predictor_call_log = pd.DataFrame(
    [
        {'call_index': i, 'origin': 'warm-up' if i < 2 else 'request', 'texts': texts}
        for i, texts in enumerate(predictor.calls)
    ]
)
predictor_call_log

,call_index,origin,texts
0,0,warm-up,"[warm up the model, second warm-up call]"
1,1,warm-up,"[I cannot log in, My invoice is wrong]"


## 7) Parse and Persist the Structured Logs

Each non-empty log line is a JSON payload. Persisting them under `reports/sample_run/serving/` makes them inspectable with the same tooling used for other notebook artifacts.

In [7]:
log_lines = [line for line in log_stream.getvalue().splitlines() if line.strip()]
parsed_logs = [json.loads(line) for line in log_lines]
LOG_DUMP_PATH.write_text('\n'.join(json.dumps(item, sort_keys=True) for item in parsed_logs) + '\n', encoding='utf-8')
print(f'{len(parsed_logs)} log lines written to {LOG_DUMP_PATH}')
pd.DataFrame(parsed_logs)

5 log lines written to C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\serving\request_log.jsonl


,client,event,latency_ms,method,model_version,path,status
0,testclient:50000,request,2.273,GET,0.5.0-demo,/health,200
1,testclient:50000,request,0.626,GET,0.5.0-demo,/health/live,200
2,testclient:50000,request,0.805,GET,0.5.0-demo,/health/ready,200
3,testclient:50000,request,2.135,POST,0.5.0-demo,/predict,200
4,testclient:50000,request,1.125,POST,0.5.0-demo,/predict,422


## 8) Readiness Failure Mode

When the predictor factory throws (missing artifact, bad config, transformers import failure) the app still starts but `/health/ready` returns 503 with a diagnostic payload. Load balancers and orchestrators can use this to route traffic only to healthy instances.

In [8]:
def broken_factory(_path: Path) -> FakePredictor:
    raise RuntimeError('model artifact not found at the expected path')


broken_app = create_app(
    model_dir=MODEL_DIR,
    predictor_factory=broken_factory,
    warm_up_texts=None,
    model_version=MODEL_VERSION,
)
with TestClient(broken_app) as broken_client:
    response = broken_client.get('/health/ready')
READINESS_FAIL_PATH.write_text(json.dumps(response.json(), indent=2), encoding='utf-8')
print(f'Status:        {response.status_code}')
print(f'Persisted to:  {READINESS_FAIL_PATH}')
response.json()

Status:        503
Persisted to:  C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\serving\readiness_failure.json


{'detail': {'status': 'not_ready',
  'startup_error': "RuntimeError('model artifact not found at the expected path')"}}

## 9) Optional: Prometheus Metrics

Set `enable_metrics=True` and the app gains a `/metrics` endpoint that exposes request counts and latency histograms in Prometheus exposition format. The dependency is optional — if `prometheus-client` is not installed the cell prints a hint and continues.

In [9]:
metrics_sample: str | None = None
try:
    metrics_app = create_app(
        model_dir=MODEL_DIR,
        predictor_factory=lambda _path: FakePredictor(),
        warm_up_texts=('warm up',),
        model_version=MODEL_VERSION,
        enable_metrics=True,
    )
except ImportError as exc:
    print(f'prometheus-client not installed; skipping metrics demo ({exc})')
else:
    with TestClient(metrics_app) as metrics_client:
        for _ in range(3):
            metrics_client.post('/predict', json={'texts': ['ping']})
        metrics_response = metrics_client.get('/metrics')
    metrics_sample = metrics_response.text
    METRICS_DUMP_PATH.write_text(metrics_sample, encoding='utf-8')
    print(f'/metrics status: {metrics_response.status_code}')
    print(f'Saved sample to: {METRICS_DUMP_PATH}')
    print()
    print('\n'.join(metrics_sample.splitlines()[:25]))

/metrics status: 200
Saved sample to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\serving\metrics_sample.txt

# HELP hf_lab_requests_total Number of API requests handled, labelled by method/path/status.
# TYPE hf_lab_requests_total counter
hf_lab_requests_total{method="POST",path="/predict",status="200"} 3.0
# HELP hf_lab_requests_created Number of API requests handled, labelled by method/path/status.
# TYPE hf_lab_requests_created gauge
hf_lab_requests_created{method="POST",path="/predict",status="200"} 1.7792077362232592e+09
# HELP hf_lab_request_latency_seconds Request latency in seconds, labelled by method/path.
# TYPE hf_lab_request_latency_seconds histogram
hf_lab_request_latency_seconds_bucket{le="0.005",method="POST",path="/predict"} 3.0
hf_lab_request_latency_seconds_bucket{le="0.01",method="POST",path="/predict"} 3.0
hf_lab_request_latency_seconds_bucket{le="0.025",method="POST",path="/predict"} 3.0
hf_lab_request_latency_seconds_bucket{le

## 10) Docker Compose Snippet

The repo ships a `docker-compose.yml` at the root that builds the image, mounts a host `artifacts/` directory read-only, and wires the container healthcheck to `/health/ready`. Use it as the starting point for a production deployment.

In [10]:
compose_path = ROOT / 'docker-compose.yml'
print(compose_path)
print('-' * 60)
print(compose_path.read_text(encoding='utf-8'))

C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\docker-compose.yml
------------------------------------------------------------
version: "3.9"

services:
  hf-lab-serve:
    build: .
    image: hf-finetuning-lab:latest
    container_name: hf-lab-serve
    environment:
      # Override these from a .env file or your secrets manager. The defaults
      # point at the example artifact baked into the image so the container
      # starts; mount a real model artifact for production.
      HF_LAB_MODEL_DIR: "/app/artifacts/models/support-triage"
      HF_LAB_MODEL_VERSION: "0.1.0"
      HF_LAB_ENABLE_METRICS: "false"
    ports:
      - "8000:8000"
    # Bind-mount a host directory holding the trained model so the image
    # itself does not need to ship one.
    volumes:
      - ./artifacts:/app/artifacts:ro
    healthcheck:
      test:
        - "CMD-SHELL"
        - "python -c \"import urllib.request; urllib.request.urlopen('http://localhost:8000/health/ready', timeout=2)\""


## 11) Checklist
- `/health/live` is a cheap liveness probe; `/health/ready` only returns 200 once the predictor is loaded and warmed up.
- The warm-up step (configurable via `warm_up_texts`) hits the predictor once on startup so the first user request avoids cold-start latency.
- Every request emits one structured JSON log line on the `hf_finetuning_lab.serving.requests` logger with method, path, status, latency, model version, and client.
- `enable_metrics=True` mounts a `/metrics` endpoint backed by `prometheus_client` for scraping into Grafana / Datadog.
- `docker-compose.yml` builds the image, mounts the host artifact directory, and uses `/health/ready` as the container healthcheck so orchestrators only route to healthy instances.
- For production: front the API with HTTPS termination, mount the model artifact from object storage rather than baking it into the image, and stream the request log into a shared sink (CloudWatch, Loki, Datadog) instead of stdout.